# V13 — Factorial training: 4 architectures × 3 arms × 3 seeds (study design v2, §4.2–4.3)
**One Kaggle session per architecture** (set `ARCH` in the first cell). GPU T4/P100, Internet ON. About 3–7 h per architecture.

Protocol, identical for all architectures:
- timm ImageNet weights, full fine-tuning, AdamW (wd 0.05), 1-epoch warm-up + cosine, ≤ 20 epochs, early stopping on val AUC (patience 5)
- batch 32, mixed precision, gradient clip 1.0, class-balanced `WeightedRandomSampler`, mild on-the-fly augmentation (borders filled with mean grey)
- **LR chosen once per architecture** from {1e-5, 3e-5, 1e-4} on arm A / seed 42 / validation AUC; the winning sweep run is reused as that factorial cell
- seeds vary head initialisation, batch order and augmentation draws; split and external samples are fixed (V11/V12)

**Resume-safe:** every finished run writes `preds/{arch}__{arm}__s{seed}.npz`. If a session dies, add this notebook's previous
output as an input and re-run: finished runs are copied and skipped.

In [ ]:
import os, sys, json, time, glob, math, random, shutil, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.v2 as T
import timm
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ARCH = "densenet121"   # run once per architecture: densenet121, convnext_tiny, swin_tiny_patch4_window7_224, vit_base_patch16_224
# one of: vit_base_patch16_224 | densenet121 | convnext_tiny | swin_tiny_patch4_window7_224
ARMS = ["raw", "roi", "lung"]
SEEDS = [42, 1337, 2024]
LR_GRID = [1e-5, 3e-5, 1e-4]
LR_FIXED = {}                     # e.g. {"densenet121": 1e-4} to skip the sweep on a resumed session
EPOCHS, PATIENCE, BATCH, WD, WARMUP_EPOCHS = 20, 5, 32, 0.05, 1
SAVE_CKPT_SEEDS = [42, 1337, 2024]   # all seeds: V14 part B needs them (ViT ≈ 3.1 GB, others < 1 GB)
TIME_BUDGET_H = float(os.environ.get("TIME_BUDGET_H", "11.0"))
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"
INPUT = os.environ.get("V13_INPUT", "/kaggle/input")
OUT = os.environ.get("V13_OUT", "/kaggle/working")
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "2"))
if QUICK_TEST:
    EPOCHS, PATIENCE, BATCH, LR_GRID, SEEDS = 1, 1, 8, [1e-4], [42, 1337]
    SAVE_CKPT_SEEDS = SEEDS
MEAN, STD, FILL = 0.4769, 0.2414, int(round(0.4769 * 255))
os.makedirs(os.path.join(OUT, "preds"), exist_ok=True); os.makedirs(os.path.join(OUT, "ckpt"), exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T0 = time.time()
print("ARCH", ARCH, "| device", device, "| torch", torch.__version__, "| timm", timm.__version__)

## 1. Load the V12 cache

In [ ]:
def find_file(name):
    for r, _, fs in os.walk(INPUT):
        if name in fs: return os.path.join(r, name)
    return None
MANIFEST = find_file("manifest_v2.csv"); assert MANIFEST, "add the V12 output as an input"
CACHE_DIR = os.path.dirname(MANIFEST)
meta = pd.read_csv(MANIFEST)
cache = {}
for c in meta.cohort.unique():
    z = np.load(os.path.join(CACHE_DIR, f"cache_{c}.npz"))
    cache[c] = {a: z[a] for a in ARMS}
    print(c, {a: cache[c][a].shape for a in ARMS})
v12 = json.load(open(os.path.join(CACHE_DIR, "v12_summary.json")))
print("split fingerprints (from V12):", v12["split_fingerprints"])

idx = {}
mi = meta[meta.cohort == "internal"]
for s in ("train", "val", "test"):
    idx[s] = mi[mi.split == s]
EXT = [c for c in ("rsna", "nih", "vindr") if c in cache]
if QUICK_TEST:
    idx = {s: d.groupby("label", group_keys=False).head(10) for s, d in idx.items()}
print({s: (len(d), int(d.label.sum())) for s, d in idx.items()}, "| external:", EXT)

## 2. Data pipeline

In [ ]:
class CXR(Dataset):
    def __init__(self, arr, rows, labels, train):
        self.arr, self.rows, self.labels, self.train = arr, np.asarray(rows), np.asarray(labels), train
        self.aug = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05), fill=FILL),
            T.ColorJitter(brightness=0.15, contrast=0.15),
        ])
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        x = torch.from_numpy(self.arr[self.rows[i]]).unsqueeze(0)            # 1×H×W uint8
        if self.train: x = self.aug(x)
        x = (x.float() / 255.0 - MEAN) / STD
        return x.expand(3, -1, -1).contiguous(), int(self.labels[i])

def _seed_worker(k):
    s = torch.initial_seed() % 2 ** 32; np.random.seed(s); random.seed(s)

def loader(cohort_arr, d, train, seed):
    ds = CXR(cohort_arr, d.row.values, d.label.values, train)
    if train:
        y = d.label.values; w = np.where(y == 1, 1.0 / max((y == 1).sum(), 1), 1.0 / max((y == 0).sum(), 1))
        g = torch.Generator().manual_seed(seed)
        smp = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(y), replacement=True, generator=g)
        return DataLoader(ds, batch_size=BATCH, sampler=smp, num_workers=NUM_WORKERS, drop_last=True,
                          worker_init_fn=_seed_worker, generator=g, pin_memory=device.type == "cuda")
    return DataLoader(ds, batch_size=BATCH * 2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, _ in dl:
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            out.append(torch.softmax(model(x.to(device, non_blocking=True)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")

## 3. One training run

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = False

def train_one(arm, seed, lr, tag=""):
    set_seed(seed)
    model = timm.create_model(ARCH, pretrained=PRETRAINED, num_classes=2, drop_rate=0.1).to(device)
    arr = cache["internal"][arm]
    dl_tr, dl_va = loader(arr, idx["train"], True, seed), loader(arr, idx["val"], False, seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    steps = EPOCHS * len(dl_tr); warm = WARMUP_EPOCHS * len(dl_tr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(warm, 1) if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    crit = nn.CrossEntropyLoss()
    best, best_state, best_ep, bad, hist = -1, None, -1, 0, []
    for ep in range(EPOCHS):
        model.train(); tl, n = 0.0, 0
        for x, y in dl_tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); n += len(y)
        va = safe_auc(idx["val"].label.values, predict(model, dl_va))
        hist.append(dict(epoch=ep + 1, train_loss=tl / max(n, 1), val_auc=va))
        print(f"   [{ARCH} {arm} s{seed} lr{lr:g}{tag}] ep {ep + 1:02d} loss {tl / max(n, 1):.4f} val AUC {va:.4f}")
        if va > best:
            best, best_ep, bad = va, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, dict(best_val_auc=best, best_epoch=best_ep, epochs_run=len(hist)), hist

def evaluate_and_save(model, arm, seed, lr, info, hist):
    res = dict(arch=ARCH, arm=arm, seed=seed, lr=lr, **info)
    blob = {}
    for name, cohort, d in [("val", "internal", idx["val"]), ("test", "internal", idx["test"])] + \
                           [(c, c, meta[meta.cohort == c]) for c in EXT]:
        p = predict(model, loader(cache[cohort][arm], d, False, seed))
        y = d.label.values
        blob[f"{name}__prob"] = p.astype(np.float32); blob[f"{name}__y"] = y.astype(np.int8)
        blob[f"{name}__id"] = d.image_id.astype(str).values
        res[f"auc_{name}"] = safe_auc(y, p)
    blob["history"] = np.array(json.dumps(hist))
    np.savez_compressed(os.path.join(OUT, "preds", f"{ARCH}__{arm}__s{seed}.npz"), **blob)
    if seed in SAVE_CKPT_SEEDS:
        torch.save({"arch": ARCH, "arm": arm, "seed": seed, "lr": lr, "state_dict": model.state_dict()},
                   os.path.join(OUT, "ckpt", f"{ARCH}__{arm}__s{seed}.pth"))
    print("   ->", {k: round(v, 4) for k, v in res.items() if k.startswith("auc_")})
    return res

## 4. Resume: copy finished runs from a previous session's output (if attached)

In [ ]:
for f in glob.glob(os.path.join(INPUT, "**", "preds", f"{ARCH}__*.npz"), recursive=True):
    dst = os.path.join(OUT, "preds", os.path.basename(f))
    if not os.path.exists(dst): shutil.copy(f, dst)
for f in glob.glob(os.path.join(INPUT, "**", "ckpt", f"{ARCH}__*.pth"), recursive=True):
    dst = os.path.join(OUT, "ckpt", os.path.basename(f))
    if not os.path.exists(dst): shutil.copy(f, dst)
prev = glob.glob(os.path.join(INPUT, "**", f"lr_sweep_{ARCH}.csv"), recursive=True)
if prev and ARCH not in LR_FIXED:
    sw = pd.read_csv(prev[0]); LR_FIXED[ARCH] = float(sw.loc[sw.val_auc.idxmax(), "lr"])
    shutil.copy(prev[0], os.path.join(OUT, f"lr_sweep_{ARCH}.csv"))
    print("LR taken from previous sweep:", LR_FIXED[ARCH])
done = lambda arm, s: os.path.exists(os.path.join(OUT, "preds", f"{ARCH}__{arm}__s{s}.npz"))

## 5. LR sweep (arm A, seed 42, validation only), then the factorial

In [ ]:
results, run_times = [], []
if ARCH in LR_FIXED:
    LR = LR_FIXED[ARCH]
else:
    sweep = []
    for lr in LR_GRID:
        t = time.time()
        model, info, hist = train_one("raw", 42, lr, tag=" sweep")
        sweep.append(dict(lr=lr, val_auc=info["best_val_auc"], best_epoch=info["best_epoch"], model=model.cpu(), info=info, hist=hist))
        torch.cuda.empty_cache()
        run_times.append(time.time() - t)
    best = max(sweep, key=lambda d: d["val_auc"]); LR = best["lr"]
    pd.DataFrame([{k: v for k, v in d.items() if k in ("lr", "val_auc", "best_epoch")} for d in sweep]).to_csv(
        os.path.join(OUT, f"lr_sweep_{ARCH}.csv"), index=False)
    print("chosen LR:", LR)
    if not done("raw", 42):
        results.append(evaluate_and_save(best["model"].to(device), "raw", 42, LR, best["info"], best["hist"]))
    del sweep, best
    torch.cuda.empty_cache()

for seed in SEEDS:
    for arm in ARMS:
        if done(arm, seed):
            print(f"skip {arm} s{seed} (already done)"); continue
        est = np.median(run_times) if run_times else 0
        if (time.time() - T0 + 1.2 * est) / 3600 > TIME_BUDGET_H:
            print(f"TIME BUDGET reached before {arm} s{seed}: save output and resume in a new session"); break
        t = time.time()
        model, info, hist = train_one(arm, seed, LR)
        results.append(evaluate_and_save(model, arm, seed, LR, info, hist))
        run_times.append(time.time() - t)
        del model; torch.cuda.empty_cache()

## 6. Summary of all finished runs for this architecture

In [ ]:
rows = []
for f in sorted(glob.glob(os.path.join(OUT, "preds", f"{ARCH}__*.npz"))):
    z = np.load(f, allow_pickle=False); _, arm, s = os.path.basename(f)[:-4].split("__")
    r = dict(arch=ARCH, arm=arm, seed=int(s[1:]))
    for k in z.files:
        if k.endswith("__prob"):
            n = k[:-6]; r[f"auc_{n}"] = safe_auc(z[f"{n}__y"], z[k])
    h = json.loads(str(z["history"])); r["best_epoch"] = int(np.argmax([e["val_auc"] for e in h]) + 1); r["epochs_run"] = len(h)
    rows.append(r)
runs = pd.DataFrame(rows); runs["lr"] = LR
runs.to_csv(os.path.join(OUT, f"runs_{ARCH}.csv"), index=False)
print(runs.round(4).to_string(index=False))
missing = [(a, s) for s in SEEDS for a in ARMS if not done(a, s)]
print("missing runs:", missing if missing else "none, factorial complete for", ARCH)

fig, axs = plt.subplots(1, len(ARMS), figsize=(7.16, 2.2), sharey=True, layout="constrained")
for ax, arm in zip(axs, ARMS):
    for f in sorted(glob.glob(os.path.join(OUT, "preds", f"{ARCH}__{arm}__*.npz"))):
        h = json.loads(str(np.load(f)["history"]))
        ax.plot([e["epoch"] for e in h], [e["val_auc"] for e in h], lw=1, label=os.path.basename(f).split("__")[2][:-4])
    ax.set_title(f"arm {arm}", loc="left", fontsize=8); ax.set_xlabel("epoch")
axs[0].set_ylabel("validation AUC"); axs[-1].legend(frameon=False, fontsize=6)
fig.savefig(os.path.join(OUT, f"fig_curves_{ARCH}.png"), dpi=200); plt.close(fig)
print(f"elapsed {(time.time() - T0) / 3600:.2f} h")